# 36 — FFT magnitudes across box kinds and positions

For `experiment-21` we record several **kinds of box** (`cardboard`, `metal`, `shoebox`,
`slat-wood`, `taped-cardboard`, `wood`), each at a handful of **positions**. Every sample has a
complex FFT (`inputs/03_fft_shifts.npz`, shape `(1, L, F, 2)`: L lasers, F freqs, 2 = x/y) and a
recovered-audio WAV.

This notebook plots the FFT magnitude spectra (same `plot_fft_magnitude` interface as notebooks 34/35,
so you can average over `laser_idx`, `xy_idx`, and now also over **position**) and lets you play the
recovered audio. Each **box kind gets its own color**; **a different shade of that color marks the
position** within the box (light = first position → dark = last).

In [1]:
import sys
sys.path.append('/Users/eitanturok/good-vibrations/src3')

import json
from pathlib import Path

import numpy as np
import pandas as pd

experiment_dir = Path(r'D:/eturok/experiment-21/data')
SAMPLES_DIR = experiment_dir / 'samples'

In [2]:
def load_metadata(sample_dir):
    """Merge the single-key dicts in metadata.jsonl into one dict."""
    meta = {}
    with open(sample_dir / 'metadata.jsonl') as f:
        for line in f:
            line = line.strip()
            if line:
                meta.update(json.loads(line))
    return meta


rows = []
for sample_dir in sorted(SAMPLES_DIR.iterdir()):
    if not sample_dir.is_dir():
        continue
    meta = load_metadata(sample_dir)
    rows.append({
        'sample_id': meta['sample_id'],
        'box': meta['box'],
        'speaker': meta.get('speaker'),
        'com': np.array(meta['com'], dtype=float),
        'is_empty_box': meta.get('is_empty_box', False),
        'fft_path': sample_dir / 'inputs' / '03_fft_shifts.npz',
        'audio_path': sample_dir / 'recovered_audio.wav',
    })

df = pd.DataFrame(rows)
# Sort by box, then by position (com x then y). Within a box, pos_idx 0 = first position.
df['com_x'] = df['com'].apply(lambda c: c[0])
df['com_y'] = df['com'].apply(lambda c: c[1])
df = df.sort_values(['box', 'com_x', 'com_y']).reset_index(drop=True)
df['pos_idx'] = df.groupby('box').cumcount()
df['n_pos'] = df.groupby('box')['box'].transform('size')
df

,sample_id,box,speaker,com,is_empty_box,fft_path,audio_path,com_x,com_y,pos_idx,n_pos
0,000009,cardboard,1,"[-1.0, -1.0]",True,D:\eturok\experiment-21\data\samples\000009\in...,D:\eturok\experiment-21\data\samples\000009\re...,-1.000000,-1.000000,0,5
1,000006,cardboard,1,"[32.231511254019296, 221.2459807073955]",False,D:\eturok\experiment-21\data\samples\000006\in...,D:\eturok\experiment-21\data\samples\000006\re...,32.231511,221.245981,1,5
2,000008,cardboard,1,"[37.311367380560135, 50.16144975288303]",False,D:\eturok\experiment-21\data\samples\000008\in...,D:\eturok\experiment-21\data\samples\000008\re...,37.311367,50.161450,2,5
3,000007,cardboard,1,"[133.69833333333332, 52.59]",False,D:\eturok\experiment-21\data\samples\000007\in...,D:\eturok\experiment-21\data\samples\000007\re...,133.698333,52.590000,3,5
4,000005,cardboard,1,"[144.6502546689304, 207.83870967741936]",False,D:\eturok\experiment-21\data\samples\000005\in...,D:\eturok\experiment-21\data\samples\000005\re...,144.650255,207.838710,4,5
5,000029,metal,1,"[-1.0, -1.0]",True,D:\eturok\experiment-21\data\samples\000029\in...,D:\eturok\experiment-21\data\samples\000029\re...,-1.000000,-1.000000,0,5
6,000028,metal,1,"[25.936465563267486, 60.333155365723435]",False,D:\eturok\experiment-21\data\samples\000028\in...,D:\eturok\experiment-21\data\samples\000028\re...,25.936466,60.333155,1,5
7,000026,metal,1,"[32.41802872709262, 209.222882615156]",False,D:\eturok\experiment-21\data\samples\000026\in...,D:\eturok\experiment-21\data\samples\000026\re...,32.418029,209.222883,2,5
8,000027,metal,1,"[60.15365733922435, 61.012272950417284]",False,D:\eturok\experiment-21\data\samples\000027\in...,D:\eturok\experiment-21\data\samples\000027\re...,60.153657,61.012273,3,5
9,000025,metal,1,"[66.19184222321829, 210.48139847601973]",False,D:\eturok\experiment-21\data\samples\000025\in...,D:\eturok\experiment-21\data\samples\000025\re...,66.191842,210.481398,4,5


## Colors: one hue per box, shaded by position

Each box kind is assigned a distinct base color. Within a box, the position index picks a shade of
that base color — the first position is the lightest tint and the last position is the full base color,
so lines from the same box cluster visually while staying individually distinguishable.

In [12]:
from plotly.colors import qualitative, hex_to_rgb

BOXES = sorted(df['box'].unique())
# One base color per box (Plotly qualitative palette has 10 colors; we have 6 boxes).
BOX_BASE = {box: qualitative.Plotly[i % len(qualitative.Plotly)] for i, box in enumerate(BOXES)}


def shade(base_hex, t):
    """Blend a base color toward white. t in [0, 1]: t=1 -> full base, t=0 -> light tint.

    We keep the blend factor in [0.3, 1] so even the lightest position stays visible.
    """
    r, g, b = hex_to_rgb(base_hex)
    f = 0.3 + 0.7 * t
    r = int(r * f + 255 * (1 - f))
    g = int(g * f + 255 * (1 - f))
    b = int(b * f + 255 * (1 - f))
    return f'rgb({r},{g},{b})'


def sample_color(box, pos_idx, n_pos):
    """Shade of the box's base color for a given position (light = first, dark = last)."""
    t = 1.0 if n_pos <= 1 else pos_idx / (n_pos - 1)
    return shade(BOX_BASE[box], t)


BOX_BASE

{'cardboard': '#636EFA',
 'metal': '#EF553B',
 'shoebox': '#00CC96',
 'slat-wood': '#AB63FA',
 'taped-cardboard': '#FFA15A',
 'wood': '#19D3F3'}

## Plot FFT magnitudes

`plot_fft_magnitude` mirrors the interface from notebook 34. Each sample is **doubly encoded**: a
shade of its box's color *and* a marker symbol, both keyed to the position index — so positions are
distinguishable by shape as well as shade. (Markers are drawn sparsely along each line for legibility;
the legend swatch shows the position's marker.)

- `boxes` — which box kinds to plot (default `None` → all). Pass a string or list of strings.
- `laser_idx` — `None` averages over all 100 lasers; an int picks one laser.
- `xy_idx` — `None` averages over the x/y directions; `0`/`1` picks one.
- `average_positions` — `False` draws one line per sample (shaded + marked by position); `True`
  averages each box's FFT magnitude over its positions and draws **one line per box** in the box's
  base color (positions collapsed, so no per-position marker).
- `empty_diff` — `True` subtracts the empty-box spectrum (the `is_empty_box` sample) from every other
  sample **in the same box**, so the empty box is zero and each line shows the change the object adds.
- `normalize` — `True` applies the pipeline's `std-sample` normalization (divide by the scalar std
  over lasers/freqs/x-y, ddof=1, floored at 1e-8; matches `_normalize_fft` in `vibrations_pipeline.py`).

**Order when both `empty_diff` and `normalize` are on: subtract first, then normalize.** Within a box
the empty and object recordings share the same scale, so subtracting in raw magnitude space cancels the
baseline exactly (empty box → 0); normalizing the *difference* afterward puts different boxes' residuals
on a common scale. Normalizing first would divide the two terms by different stds and leave a baseline
residual. In `average_positions` mode the empty box is excluded from the per-box average.

In [16]:
import plotly.graph_objects as go

# A distinct marker symbol per position index (within a box). Combined with the color shade,
# each position is doubly encoded: shade of the box color AND a marker shape.
POS_MARKERS = ['circle', 'square', 'diamond', 'triangle-up', 'star', 'cross', 'x', 'pentagon']


def pos_marker(pos_idx):
    return POS_MARKERS[pos_idx % len(POS_MARKERS)]


def load_full_mag(path):
    """Raw |fft| magnitude (no normalization) and the frequency axis. Shape (L, F, 2)."""
    with np.load(path) as d:
        fft = d['fft']      # (1, L, F, 2) complex
        freqs = d['freqs']  # (F,) Hz
    return freqs, np.abs(fft[0].astype(np.complex128)).astype(np.float32)


def _std_sample(mag):
    """std-sample scale: scalar std over all of (L, F, 2), ddof=1, floored at 1e-8.

    Matches _normalize_fft('std-sample') in vibrations_pipeline.py.
    """
    return max(float(mag.astype(np.float64).std(ddof=1)), 1e-8)


def _reduce(mag, laser_idx, xy_idx):
    """(L, F, 2) -> (F,). laser_idx/xy_idx=None averages that axis, else selects an index."""
    mag = mag.mean(axis=0) if laser_idx is None else mag[laser_idx]  # (F, 2)
    mag = mag.mean(axis=1) if xy_idx is None else mag[:, xy_idx]     # (F,)
    return mag


def sample_spectrum(path, laser_idx=None, xy_idx=None, normalize=False, empty_full=None):
    """Reduced 1D spectrum for one sample.

    Order of operations: **subtract the empty box first, then normalize.** Within a box the
    empty and object recordings share the same scale, so subtracting in raw magnitude space
    cleanly cancels the baseline (and leaves the empty box exactly at zero); applying
    std-sample normalization to the *difference* afterward puts different boxes' residuals on
    a common scale. (Normalizing first would divide the two terms by different stds, so the
    shared baseline would not cancel.) empty_full is the raw (L, F, 2) empty-box magnitude,
    or None to skip the subtraction.
    """
    freqs, mag = load_full_mag(path)
    if empty_full is not None:
        mag = mag - empty_full          # empty_diff: raw subtraction, BEFORE normalize
    if normalize:
        mag = mag / _std_sample(mag)    # std-sample on the (possibly differenced) signal
    return freqs, _reduce(mag, laser_idx, xy_idx)


def _as_list(x):
    if x is None:
        return None
    return [x] if isinstance(x, str) else list(x)


def _empty_full_mag(box):
    """Raw (L, F, 2) magnitude of the box's empty-box (is_empty_box) sample."""
    erow = df[(df['box'] == box) & df['is_empty_box']]
    if len(erow) == 0:
        raise ValueError(f'empty_diff: no is_empty_box sample for box {box!r}')
    _, mag = load_full_mag(erow.iloc[0]['fft_path'])
    return mag


def _add_sample_trace(fig, freqs, mag, color, symbol, name, box, n_markers=15):
    """One sample = a full line plus a few sparse markers (the marker carries the legend).

    Putting a marker on all ~3400 points would be unreadable, so the line is drawn over the
    full spectrum (no legend) and a downsampled markers-only trace supplies the legend entry,
    whose swatch shows the position's marker symbol.
    """
    fig.add_trace(go.Scatter(x=freqs, y=mag, mode='lines',
                             line=dict(color=color, width=1),
                             legendgroup=name, showlegend=False, hoverinfo='skip'))
    step = max(1, len(freqs) // n_markers)
    fig.add_trace(go.Scatter(x=freqs[::step], y=mag[::step], mode='markers', name=name,
                             marker=dict(color=color, symbol=symbol, size=7,
                                         line=dict(color='rgba(0,0,0,0.4)', width=0.5)),
                             legendgroup=name, showlegend=True))


def plot_fft_magnitude(boxes=None, laser_idx=None, xy_idx=None, average_positions=False,
                       normalize=False, empty_diff=False):
    boxes = _as_list(boxes)
    sub = df if boxes is None else df[df['box'].isin(boxes)]

    # Raw empty-box magnitude per box, subtracted before normalization (empty stays {} otherwise).
    empty_full = ({box: _empty_full_mag(box) for box in sub['box'].unique()}
                  if empty_diff else {})

    fig = go.Figure()

    if average_positions:
        # One line per box: average the (reduced) FFT magnitude over that box's positions.
        # Positions are collapsed, so there is no per-position marker here.
        for box, grp in sub.groupby('box'):
            mags = []
            for s in grp.itertuples():
                if empty_diff and s.is_empty_box:
                    continue  # the empty box is the zero reference, not a position to average
                freqs, mag = sample_spectrum(s.fft_path, laser_idx, xy_idx, normalize,
                                             empty_full.get(box))
                mags.append(mag)
            mag = np.mean(mags, axis=0)
            fig.add_trace(go.Scatter(x=freqs, y=mag, mode='lines', name=box,
                                     line=dict(color=BOX_BASE[box], width=1.8)))
    else:
        # One line per sample: box color shaded by position, marker symbol by position.
        for s in sub.itertuples():
            freqs, mag = sample_spectrum(s.fft_path, laser_idx, xy_idx, normalize,
                                         empty_full.get(s.box))
            color = sample_color(s.box, s.pos_idx, s.n_pos)
            name = f'{s.box} [{s.sample_id}] (x={s.com[0]:.0f}, y={s.com[1]:.0f})'
            _add_sample_trace(fig, freqs, mag, color, pos_marker(s.pos_idx), name, s.box)

    laser_str = 'avg' if laser_idx is None else laser_idx
    xy_str = 'avg' if xy_idx is None else xy_idx
    flags = (['avg over positions' if average_positions else 'per position']
             + (['empty-diff'] if empty_diff else [])
             + (['normalized'] if normalize else []))
    fig.update_layout(
        title=f'FFT magnitude (experiment-21) | laser={laser_str}, xy={xy_str}, {", ".join(flags)}',
        xaxis_title='frequency (Hz)',
        yaxis_title='FFT magnitude' + (' (minus empty box)' if empty_diff else ''),
        legend_title='box [sample] (com)', width=1200, height=600)
    fig.show()


plot_fft_magnitude(normalize=True)

In [17]:
plot_fft_magnitude(normalize=True, boxes='metal')

In [18]:
plot_fft_magnitude(normalize=True, boxes='wood')

In [22]:
plot_fft_magnitude(normalize=True, boxes='slat-wood')

In [20]:
plot_fft_magnitude(normalize=True, boxes='taped-cardboard')

In [21]:
plot_fft_magnitude(normalize=True, boxes='shoebox')

A few variations. Average over positions to compare box kinds directly, or drill into a single laser /
single direction. These mirror notebook 34's averaging knobs.

In [ ]:
# One line per box, averaged over its positions.
plot_fft_magnitude(average_positions=True)

# std-sample normalized (puts every box's spectrum on a common scale).
plot_fft_magnitude(normalize=True)

# Object signal only: subtract each box's empty-box spectrum (empty box sits at zero).
plot_fft_magnitude(empty_diff=True)

# Normalized + empty-diff, averaged per box.
# plot_fft_magnitude(average_positions=True, normalize=True, empty_diff=True)

# A single laser, x-direction only, for two box kinds.
# plot_fft_magnitude(boxes=['wood', 'metal'], laser_idx=5, xy_idx=0)

## Play recovered audio

Same filtering as the FFT plot: `play_recovered_audio(boxes=None)` walks the selected samples and
shows a labeled audio player for each `recovered_audio.wav`, grouped by box and ordered by position
(matching the FFT shading: first position first).

In [23]:
from IPython.display import Audio, display, HTML


def play_recovered_audio(boxes=None):
    boxes = _as_list(boxes)
    sub = df if boxes is None else df[df['box'].isin(boxes)]
    for box, grp in sub.groupby('box'):
        display(HTML(f'<h3 style="color:{BOX_BASE[box]}">{box}</h3>'))
        for s in grp.itertuples():
            color = sample_color(s.box, s.pos_idx, s.n_pos)
            label = (f'<span style="color:{color}">&#9632;</span> '
                     f'<b>{s.sample_id}</b> &nbsp; x={s.com[0]:.0f}, y={s.com[1]:.0f}')
            display(HTML(label))
            display(Audio(filename=str(s.audio_path)))


play_recovered_audio()

## Normalized cross-correlation (NCC) between positions

Same NCC as notebook 34: for two complex FFTs, `|<a, b>| / (||a|| ||b||)`, the magnitude of the
Hermitian inner product over the full `(L, F, 2)` array normalized by the L2 norms — a similarity in
`[0, 1]` that is exactly 1 for a sample with itself.

We compute it **pairwise within each box** (positions of the same box are comparable; different boxes
are different objects). Then:

1. a **5×5 NCC heatmap per box** (the 5 samples = empty box + 4 positions), with each box's
   **off-diagonal NCC sum** (sum over the unique position pairs = the lower-triangle cells shown) in
   its subplot title; and
2. a scatter of **NCC vs COM distance** over the valid within-box pairs (empty-box pairs have no COM,
   so they're dropped), annotated with **Pearson r / r²** (linear) and **Spearman ρ** (monotonic) —
   both signed, so they tell you whether NCC tends to rise or fall as two positions move apart.

In [24]:
from itertools import combinations

from scipy.stats import pearsonr, spearmanr


def normalized_cross_correlation(fft_a, fft_b):
    """NCC of two complex FFTs: |<a, b>| / (||a|| * ||b||), a similarity in [0, 1].

    Hermitian inner product sum(conj(a) * b) over the full (L, F, 2) array, normalized by the
    product of the L2 norms. Computed in complex128 so a sample's NCC with itself is 1.0.
    """
    a = np.asarray(fft_a, dtype=np.complex128).ravel()
    b = np.asarray(fft_b, dtype=np.complex128).ravel()
    return np.abs(np.vdot(a, b)) / (np.linalg.norm(a) * np.linalg.norm(b))


def load_fft(path):
    with np.load(path) as d:
        return d['fft']  # (1, L, F, 2) complex


# Load each sample's FFT once (loading is the expensive part), then compute NCC for every
# within-box pair of samples. COM distance is only defined for valid pairs (neither is the
# empty box, whose com is the [-1, -1] sentinel).
ffts = {s.sample_id: load_fft(s.fft_path) for s in df.itertuples()}

pair_rows = []
for box, grp in df.groupby('box'):
    for s1, s2 in combinations(grp.itertuples(), 2):
        valid = not (s1.is_empty_box or s2.is_empty_box)
        com_dist = float(np.linalg.norm(s1.com - s2.com)) if valid else np.nan
        pair_rows.append({
            'box': box,
            'sample_id1': s1.sample_id, 'sample_id2': s2.sample_id,
            'pos_idx1': s1.pos_idx, 'pos_idx2': s2.pos_idx,
            'com1': s1.com, 'com2': s2.com,
            'is_empty1': s1.is_empty_box, 'is_empty2': s2.is_empty_box,
            'valid': valid, 'com_dist': com_dist,
            'ncc': normalized_cross_correlation(ffts[s1.sample_id], ffts[s2.sample_id]),
        })

pair_df = pd.DataFrame(pair_rows)
pair_df

,box,sample_id1,sample_id2,pos_idx1,pos_idx2,com1,com2,is_empty1,is_empty2,valid,com_dist,ncc
0,cardboard,000009,000006,0,1,"[-1.0, -1.0]","[32.231511254019296, 221.2459807073955]",True,False,False,NaN,0.209297
1,cardboard,000009,000008,0,2,"[-1.0, -1.0]","[37.311367380560135, 50.16144975288303]",True,False,False,NaN,0.163332
2,cardboard,000009,000007,0,3,"[-1.0, -1.0]","[133.69833333333332, 52.59]",True,False,False,NaN,0.193415
3,cardboard,000009,000005,0,4,"[-1.0, -1.0]","[144.6502546689304, 207.83870967741936]",True,False,False,NaN,0.075463
4,cardboard,000006,000008,1,2,"[32.231511254019296, 221.2459807073955]","[37.311367380560135, 50.16144975288303]",False,False,True,171.159930,0.895828
5,cardboard,000006,000007,1,3,"[32.231511254019296, 221.2459807073955]","[133.69833333333332, 52.59]",False,False,True,196.825699,0.960180
6,cardboard,000006,000005,1,4,"[32.231511254019296, 221.2459807073955]","[144.6502546689304, 207.83870967741936]",False,False,True,113.215409,0.051356
7,cardboard,000008,000007,2,3,"[37.311367380560135, 50.16144975288303]","[133.69833333333332, 52.59]",False,False,True,96.417556,0.956188
8,cardboard,000008,000005,2,4,"[37.311367380560135, 50.16144975288303]","[144.6502546689304, 207.83870967741936]",False,False,True,190.745262,0.051624
9,cardboard,000007,000005,3,4,"[133.69833333333332, 52.59]","[144.6502546689304, 207.83870967741936]",False,False,True,155.634528,0.049722


In [25]:
from plotly.subplots import make_subplots

# One 5x5 NCC heatmap per box (the box's 5 samples = empty box + 4 positions). The matrix is
# symmetric and the diagonal is self-NCC = 1, so we blank the upper triangle and show only the
# unique pairs (lower triangle). Each subplot title reports that box's off-diagonal NCC sum =
# sum of NCC over the unique position pairs (exactly the cells shown).
n_cols = 3
n_rows = int(np.ceil(len(BOXES) / n_cols))

# Precompute every box's matrix, tick labels, and off-diagonal sum first (titles need the sums).
box_mat, box_labels, box_offsum = {}, {}, {}
for box in BOXES:
    grp = df[df['box'] == box].sort_values('pos_idx')
    ids = list(grp['sample_id'])
    m = len(ids)
    mat = np.eye(m)
    for r in pair_df[pair_df['box'] == box].itertuples():
        i, j = ids.index(r.sample_id1), ids.index(r.sample_id2)
        mat[i, j] = mat[j, i] = r.ncc
    mat[np.triu(np.ones((m, m), dtype=bool), k=1)] = np.nan  # blank upper triangle (duplicates)
    box_mat[box] = mat
    box_labels[box] = ['empty' if s.is_empty_box
                       else f'p{s.pos_idx}<br>({s.com[0]:.0f},{s.com[1]:.0f})'
                       for s in grp.itertuples()]
    box_offsum[box] = pair_df[pair_df['box'] == box]['ncc'].sum()

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f'{box} | off-diag sum = {box_offsum[box]:.2f}' for box in BOXES],
    horizontal_spacing=0.09, vertical_spacing=0.13)

for k, box in enumerate(BOXES):
    row, col = k // n_cols + 1, k % n_cols + 1
    z = box_mat[box]
    labels = box_labels[box]
    m = len(labels)
    text = [['' if np.isnan(z[i, j]) else f'{z[i, j]:.2f}' for j in range(m)] for i in range(m)]
    fig.add_trace(go.Heatmap(
        z=z, x=list(range(m)), y=list(range(m)), zmin=0, zmax=1, colorscale='Blues',
        text=text, texttemplate='%{text}', textfont=dict(size=8),
        showscale=(k == 0), colorbar=dict(title='NCC', len=0.9), hoverongaps=False),
        row=row, col=col)
    fig.update_xaxes(tickvals=list(range(m)), ticktext=labels, tickangle=90,
                     row=row, col=col)
    fig.update_yaxes(tickvals=list(range(m)), ticktext=labels, autorange='reversed',
                     scaleanchor=f'x{k + 1}', scaleratio=1, row=row, col=col)

fig.update_layout(title='Pairwise NCC within each box (empty box + positions)',
                  width=1250, height=420 * n_rows)
fig.show()

In [26]:
# NCC vs COM distance, pooling the valid within-box position pairs (empty-box pairs have no
# COM, so they're dropped). Pearson r (and r^2) measures linear association; Spearman rho
# measures monotonic association ("as positions move apart, does NCC consistently go up/down?").
# Both are signed, so their sign is the direction. Points colored by box; dashed line = the
# pooled least-squares fit.
valid = pair_df[pair_df['valid']].copy()
x = valid['com_dist'].to_numpy()
y = valid['ncc'].to_numpy()
r = pearsonr(x, y).statistic
rho = spearmanr(x, y).statistic

fig = go.Figure()
for box, grp in valid.groupby('box'):
    hover = [f'{g.box}<br>p{g.pos_idx1} vs p{g.pos_idx2}<br>'
             f'com_dist={g.com_dist:.1f}, ncc={g.ncc:.3f}' for g in grp.itertuples()]
    fig.add_trace(go.Scatter(
        x=grp['com_dist'], y=grp['ncc'], mode='markers', name=box,
        marker=dict(color=BOX_BASE[box], size=9, line=dict(color='rgba(0,0,0,0.4)', width=0.5)),
        text=hover, hoverinfo='text'))

# Pooled least-squares trend line over all valid pairs.
slope, intercept = np.polyfit(x, y, 1)
xs = np.array([x.min(), x.max()])
fig.add_trace(go.Scatter(x=xs, y=intercept + slope * xs, mode='lines', name='linear fit',
                         line=dict(color='black', dash='dash', width=1.5)))

fig.update_layout(
    title=(f'NCC vs COM distance (within-box pairs) | '
           f'Pearson r = {r:.3f} (r² = {r**2:.3f}), Spearman ρ = {rho:.3f}'),
    xaxis_title='COM distance (px)', yaxis_title='NCC',
    legend_title='box', width=950, height=550)
fig.show()

### NCC vs COM distance, per box

The pooled scatter above mixes all boxes. Here we split it out: **one scatter per box**, each with its
own least-squares trend line and its own **Pearson r / r²** (linear) and **Spearman ρ** (monotonic),
computed only over that box's valid position pairs — the same per-box split used for the NCC heatmaps.
Each box has 4 positions → C(4,2) = 6 pairs, so these correlations are based on few points and are noisy;
read them as a rough per-material trend, not a precise estimate.

In [27]:
def _corr(x, y):
    """Pearson r and Spearman rho, guarded for the degenerate cases (need >=3 points and some
    variance in both axes, else the coefficients are undefined). Returns (r, rho), NaN if undefined."""
    if len(x) < 3 or np.ptp(x) == 0 or np.ptp(y) == 0:
        return np.nan, np.nan
    return pearsonr(x, y).statistic, spearmanr(x, y).statistic


# One NCC-vs-COM-distance scatter per box, each with its own trend line and its own r/r^2/rho
# over that box's valid position pairs only.
n_cols = 3
n_rows = int(np.ceil(len(BOXES) / n_cols))

box_corr = {}  # box -> (r, rho) for reference
titles = []
for box in BOXES:
    g = pair_df[(pair_df['box'] == box) & pair_df['valid']]
    r, rho = _corr(g['com_dist'].to_numpy(), g['ncc'].to_numpy())
    box_corr[box] = (r, rho)
    r2 = r ** 2 if np.isfinite(r) else np.nan
    titles.append(f'{box}<br>r²={r2:.3f}, ρ={rho:.3f}')

fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=titles,
                    horizontal_spacing=0.08, vertical_spacing=0.16)

for k, box in enumerate(BOXES):
    row, col = k // n_cols + 1, k % n_cols + 1
    g = pair_df[(pair_df['box'] == box) & pair_df['valid']]
    xg, yg = g['com_dist'].to_numpy(), g['ncc'].to_numpy()
    hover = [f'p{t.pos_idx1} vs p{t.pos_idx2}<br>com_dist={t.com_dist:.1f}, ncc={t.ncc:.3f}'
             for t in g.itertuples()]
    fig.add_trace(go.Scatter(
        x=xg, y=yg, mode='markers', name=box, legendgroup=box,
        marker=dict(color=BOX_BASE[box], size=10, line=dict(color='rgba(0,0,0,0.4)', width=0.5)),
        text=hover, hoverinfo='text'), row=row, col=col)
    if len(xg) >= 2 and np.ptp(xg) > 0:  # per-box least-squares trend line
        slope, intercept = np.polyfit(xg, yg, 1)
        xs = np.array([xg.min(), xg.max()])
        fig.add_trace(go.Scatter(x=xs, y=intercept + slope * xs, mode='lines',
                                 line=dict(color=BOX_BASE[box], dash='dash', width=1.5),
                                 legendgroup=box, showlegend=False, hoverinfo='skip'),
                      row=row, col=col)
    fig.update_xaxes(title_text='COM distance (px)', row=row, col=col)
    fig.update_yaxes(title_text='NCC', row=row, col=col)

fig.update_layout(title='NCC vs COM distance, per box', showlegend=False,
                  width=1250, height=420 * n_rows)
fig.show()

box_corr

{'cardboard': (0.013178541687575124, 0.3142857142857143),
 'metal': (-0.20785098327919602, -0.4285714285714286),
 'shoebox': (0.007535808401799519, 0.2571428571428572),
 'slat-wood': (-0.03964244735094519, -0.2571428571428572),
 'taped-cardboard': (-0.0861051451188084, 0.14285714285714288),
 'wood': (0.7484585476833587, 0.8285714285714287)}